In [1]:
import os
import pandas as pd
import numpy as np

%run data_loading.ipynb

In [2]:
accounts['account_type'].unique()

array(['SAVINGS', 'CHECKING', 'CREDIT CARD', 'LINE OF CREDIT',
       'MONEYMARKET', 'LOAN', 'MONEY MARKET', 'ROTH', 'MORTGAGE',
       'RETIREMENT', 'PREPAID', 'BROKERAGE', 'CONSUMER', 'CD', 'IRA',
       'AUTO', 'STUDENT', 'HSA', 'CASH MANAGEMENT', 'OTHER', '401K',
       'STOCK PLAN', 'OVERDRAFT', 'HOME EQUITY'], dtype=object)

# Feature based on Monthly Cash Flow 

In [4]:
transactions['month_period'] = transactions['posted_date'].dt.to_period('M')

datetime64[ns]


In [8]:
# DEPOSIT, PAYCHECK, REFUND, INVESTMENT_INCOME, OTHER_BENEFITS, UNEMPLOYMENT_BENEFITS
# income_categories = [2, 3, 6, 7, 8, 9]
# monthly_income = transactions[transactions['category'].isin(income_categories)].groupby('prism_consumer_id')['amount'].sum().reset_index(name='total_income')

monthly_income = transactions[transactions['credit_or_debit'] == 'CREIDT'].groupby('prism_consumer_id')['amount'].sum().reset_index(name='total_income')
monthly_spending = transactions[transactions['credit_or_debit'] == 'DEBIT'].groupby('prism_consumer_id')['amount'].sum().reset_index(name='total_spending')

transactions['posted_date'] = pd.to_datetime(transactions['posted_date'], errors='coerce')
months_active = transactions.groupby('prism_consumer_id')['posted_date'].apply(
    lambda x: x.dt.to_period('M').nunique()
).reset_index(name='month_count')

df_features = consumers[['prism_consumer_id']].merge(monthly_income, on='prism_consumer_id', how='left')
df_features = df_features.merge(monthly_spending, on='prism_consumer_id', how='left')
df_features = df_features.merge(months_active, on='prism_consumer_id', how='left')

df_features = df_features.fillna(0)

df_features['monthly_net_cash_flow'] = (
    (df_features['total_income'] - df_features['total_spending']) / df_features['month_count']
).replace([np.inf, -np.inf], 0).fillna(0)

df_features[['prism_consumer_id', 'monthly_net_cash_flow']].head()

,prism_consumer_id,monthly_net_cash_flow
0,0,-2129.772857
1,1,-3299.767143
2,2,-3190.654286
3,3,-2835.144286
4,4,-2501.387143


In [9]:
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression

df_eval = df_features.merge(consumers[['prism_consumer_id', 'DQ_TARGET']], on='prism_consumer_id')
df_eval = df_eval.dropna(subset=['DQ_TARGET'])

X = df_eval[['monthly_net_cash_flow']]
y = df_eval['DQ_TARGET']

clf = LogisticRegression()
clf.fit(X, y)

y_pred_proba = clf.predict_proba(X)[:, 1]

# Area Under Curve
auc_score = roc_auc_score(y, y_pred_proba)
print(f"AUC Score: {auc_score:.4f}")

probs_bad = y_pred_proba[y == 1]
probs_good = y_pred_proba[y == 0]

# Kolmogorov-Smirnov
ks_statistic, p_value = ks_2samp(probs_bad, probs_good)
print(f"KS Statistic: {ks_statistic:.4f}")

AUC Score: 0.5291
KS Statistic: 0.0618


# Feature based on overdraft 

In [ ]:
category_mapping[category_mapping['category']=='OVERDRAFT']

In [ ]:
overdraft_txns = transactions[transactions['category']==25]

In [ ]:
# 2. Count the number of overdrafts per consumer
# .size() counts the rows for each consumer
overdraft_counts = overdraft_txns.groupby('prism_consumer_id').size().reset_index(name='overdraft_count')

# 3. Merge this count back to your main 'consumers' dataframe
# Use how='left' to keep all consumers, even those who didn't appear in the overdraft list
df_features = consumers.merge(overdraft_counts, on='prism_consumer_id', how='left')

# 4. Fill missing values with 0
# If a consumer wasn't in the overdraft list, the merge creates a NaN. We convert that to 0.
df_features['overdraft_count'] = df_features['overdraft_count'].fillna(0).astype(int)

# View the result
df_features[['prism_consumer_id', 'overdraft_count']]

In [ ]:
df_features[df_features['overdraft_count']>0]